# Library Management System

**Python OOP Project: Task 21 (OOP Project)**

Author: **Mohammad Al Mukadam**

## Implementation
This project is implemented as a **Jupyter Notebook**.
(A multi-file implementation is considered a bonus and is not included here.)

## OOP Concepts Demonstrated
-  Classes & Objects
-  Encapsulation
-  Abstraction
-  Inheritance
-  Polymorphism
-  Constructors
-  Exception Handling
-  Clean, readable, documented code

## Project Features
- Add `Book` and `DVD` items to the library (inheritance from an abstract `LibraryItem`)
- Register members and let them borrow / return items
- Encapsulated member balances (fines) with controlled access via properties
- Polymorphic `describe()` behavior for every item type
- Custom exceptions for invalid operations (item not found, item unavailable, member not found)
- A simple class-level counter tracking how many items exist across the whole library


## 1. Custom Exceptions

Dedicated exception classes make error handling explicit and readable instead of relying on generic exceptions.

In [1]:
class LibraryError(Exception):
    """Base exception for all library-related errors."""
    pass


class ItemNotFoundError(LibraryError):
    """Raised when a requested item does not exist in the library."""
    pass


class ItemNotAvailableError(LibraryError):
    """Raised when trying to borrow an item that is already checked out."""
    pass


class MemberNotFoundError(LibraryError):
    """Raised when a member ID is not registered in the library."""
    pass


## 2. Abstraction — `LibraryItem` (Abstract Base Class)

`LibraryItem` defines a common contract that every kind of item in the library must follow. It cannot be instantiated directly — only its subclasses can. This is **abstraction**: users of the class only need to know *what* `describe()` and `checkout_period_days()` do, not *how* each subclass implements them.

In [2]:
from abc import ABC, abstractmethod


class LibraryItem(ABC):
    """Abstract base class representing any item the library can lend."""

    _total_items = 0          # static / class member — shared by ALL items

    def __init__(self, item_id: str, title: str):
        # Constructor: runs automatically whenever a LibraryItem subclass is created
        self._item_id = item_id            # protected attribute (encapsulation)
        self._title = title
        self._is_checked_out = False
        LibraryItem._total_items += 1      # update the shared class-level counter

    # --- Encapsulation: controlled read-only access via properties ---
    @property
    def item_id(self):
        return self._item_id

    @property
    def title(self):
        return self._title

    @property
    def is_checked_out(self):
        return self._is_checked_out

    def checkout(self):
        if self._is_checked_out:
            raise ItemNotAvailableError(f"'{self._title}' is already checked out.")
        self._is_checked_out = True

    def return_item(self):
        self._is_checked_out = False

    @abstractmethod
    def describe(self) -> str:
        """Every subclass MUST provide its own description (polymorphism)."""
        raise NotImplementedError

    @abstractmethod
    def checkout_period_days(self) -> int:
        """Every subclass MUST define its own loan period."""
        raise NotImplementedError

    @classmethod
    def total_items(cls) -> int:
        """Static/class-level access — total items across the whole library."""
        return cls._total_items

    def __repr__(self):
        status = "checked out" if self._is_checked_out else "available"
        return f"<{self.__class__.__name__} '{self._title}' ({status})>"


## 3. Inheritance & Polymorphism — `Book` and `DVD`

Both classes **inherit** from `LibraryItem` and reuse its shared logic (`checkout`, `return_item`, the class counter), while **overriding** `describe()` and `checkout_period_days()` with their own behavior — this is **run-time polymorphism**: the same method call (`item.describe()`) produces different results depending on the actual object type.

In [3]:
class Book(LibraryItem):
    def __init__(self, item_id: str, title: str, author: str, pages: int):
        super().__init__(item_id, title)     # calls the parent constructor
        self.author = author
        self.pages = pages

    def describe(self) -> str:               # method overriding
        return f"📖 Book: '{self.title}' by {self.author} ({self.pages} pages)"

    def checkout_period_days(self) -> int:    # method overriding
        return 21   # books can be borrowed for 3 weeks


class DVD(LibraryItem):
    def __init__(self, item_id: str, title: str, runtime_minutes: int):
        super().__init__(item_id, title)
        self.runtime_minutes = runtime_minutes

    def describe(self) -> str:               # method overriding
        return f"💿 DVD: '{self.title}' ({self.runtime_minutes} min)"

    def checkout_period_days(self) -> int:    # method overriding
        return 7    # DVDs can be borrowed for 1 week


## 4. Encapsulation — `Member`

A library `Member` keeps its balance (fines owed) **private**. External code cannot set an arbitrary/negative balance directly — it must go through controlled methods (`add_fine`, `pay_fine`), which is the essence of encapsulation.

In [4]:
class Member:
    def __init__(self, member_id: str, name: str):
        self.member_id = member_id
        self.name = name
        self.__balance = 0.0                  # private attribute (name-mangled)
        self.borrowed_items = []              # composition: Member "has" borrowed items

    @property
    def balance(self):
        """Read-only public access to the private balance."""
        return self.__balance

    def add_fine(self, amount: float):
        if amount < 0:
            raise ValueError("Fine amount cannot be negative.")
        self.__balance += amount

    def pay_fine(self, amount: float):
        if amount < 0:
            raise ValueError("Payment amount cannot be negative.")
        self.__balance = max(0.0, self.__balance - amount)

    def __repr__(self):
        return f"<Member {self.name} ({self.member_id}) - balance: ${self.__balance:.2f}>"


## 5. Composition — `Library`

`Library` **has** a collection of `LibraryItem`s and `Member`s (composition, a "has-a" relationship) rather than inheriting from them. It coordinates borrowing/returning and raises the custom exceptions defined above whenever something goes wrong.

In [5]:
class Library:
    def __init__(self, name: str):
        self.name = name
        self._items = {}      # item_id -> LibraryItem   (composition)
        self._members = {}    # member_id -> Member       (composition)

    def add_item(self, item: LibraryItem):
        self._items[item.item_id] = item

    def register_member(self, member: Member):
        self._members[member.member_id] = member

    def _get_item(self, item_id: str) -> LibraryItem:
        try:
            return self._items[item_id]
        except KeyError:
            raise ItemNotFoundError(f"No item found with id '{item_id}'.")

    def _get_member(self, member_id: str) -> Member:
        try:
            return self._members[member_id]
        except KeyError:
            raise MemberNotFoundError(f"No member found with id '{member_id}'.")

    def borrow_item(self, member_id: str, item_id: str):
        member = self._get_member(member_id)
        item = self._get_item(item_id)

        item.checkout()                       # may raise ItemNotAvailableError
        member.borrowed_items.append(item)
        print(f"✅ {member.name} borrowed '{item.title}' "
              f"for {item.checkout_period_days()} days.")

    def return_item(self, member_id: str, item_id: str):
        member = self._get_member(member_id)
        item = self._get_item(item_id)

        if item not in member.borrowed_items:
            raise LibraryError(f"{member.name} did not borrow '{item.title}'.")

        item.return_item()
        member.borrowed_items.remove(item)
        print(f"📥 {member.name} returned '{item.title}'.")

    def catalog(self):
        """Polymorphism in action: describe() behaves differently per item type."""
        for item in self._items.values():
            print(item.describe())


## 6. Demo — Putting It All Together

Create a library, add items (a `Book` and a `DVD`), register members, and exercise the normal borrow/return flow.

In [6]:
library = Library("City Central Library")

# Adding items (constructors run here)
book1 = Book("B001", "Clean Code", "Robert C. Martin", 464)
book2 = Book("B002", "The Pragmatic Programmer", "Andrew Hunt", 352)
dvd1  = DVD("D001", "The Matrix", 136)

for item in (book1, book2, dvd1):
    library.add_item(item)

# Registering members
alice = Member("M001", "Alice")
bob   = Member("M002", "Bob")
library.register_member(alice)
library.register_member(bob)

print("--- Catalog (polymorphic describe()) ---")
library.catalog()

print("\n--- Borrowing ---")
library.borrow_item("M001", "B001")   # Alice borrows Clean Code
library.borrow_item("M002", "D001")   # Bob borrows The Matrix

print("\n--- Total items ever created (class-level / static counter) ---")
print(LibraryItem.total_items())


--- Catalog (polymorphic describe()) ---
📖 Book: 'Clean Code' by Robert C. Martin (464 pages)
📖 Book: 'The Pragmatic Programmer' by Andrew Hunt (352 pages)
💿 DVD: 'The Matrix' (136 min)

--- Borrowing ---
✅ Alice borrowed 'Clean Code' for 21 days.
✅ Bob borrowed 'The Matrix' for 7 days.

--- Total items ever created (class-level / static counter) ---
3


## 7. Exception Handling in Action

The custom exceptions defined earlier are triggered here to show real error handling rather than the program crashing.

In [7]:
# 1) Trying to borrow an item that is already checked out
try:
    library.borrow_item("M002", "B001")     # Clean Code is already borrowed by Alice
except ItemNotAvailableError as e:
    print(f"❌ ItemNotAvailableError: {e}")

# 2) Trying to borrow an item that doesn't exist
try:
    library.borrow_item("M001", "B999")
except ItemNotFoundError as e:
    print(f"❌ ItemNotFoundError: {e}")

# 3) Trying to act on a member that isn't registered
try:
    library.borrow_item("M999", "B002")
except MemberNotFoundError as e:
    print(f"❌ MemberNotFoundError: {e}")

# 4) Encapsulation guard: negative fine is rejected
try:
    alice.add_fine(-5)
except ValueError as e:
    print(f"❌ ValueError: {e}")

# Applying a valid fine and paying part of it
alice.add_fine(12.50)
alice.pay_fine(2.50)
print(f"\n💰 {alice}")


❌ ItemNotAvailableError: 'Clean Code' is already checked out.
❌ ItemNotFoundError: No item found with id 'B999'.
❌ MemberNotFoundError: No member found with id 'M999'.
❌ ValueError: Fine amount cannot be negative.

💰 <Member Alice (M001) - balance: $10.00>


## 8. Returning Items

In [8]:
library.return_item("M001", "B001")
library.return_item("M002", "D001")

print("\n--- Catalog after returns ---")
library.catalog()


📥 Alice returned 'Clean Code'.
📥 Bob returned 'The Matrix'.

--- Catalog after returns ---
📖 Book: 'Clean Code' by Robert C. Martin (464 pages)
📖 Book: 'The Pragmatic Programmer' by Andrew Hunt (352 pages)
💿 DVD: 'The Matrix' (136 min)


## Summary

This notebook implemented a small but complete **Library Management System** and, along
the way, demonstrated every required OOP concept:

| Concept | Where it appears |
|---|---|
| Classes & Objects | `Book`, `DVD`, `Member`, `Library`, and their instances |
| Encapsulation | `Member.__balance` (private) exposed only via controlled methods/properties |
| Abstraction | `LibraryItem` abstract base class hides implementation details behind `describe()` |
| Inheritance | `Book` and `DVD` inherit from `LibraryItem` |
| Polymorphism | `library.catalog()` calls `describe()` and gets different results per item type |
| Constructors | `__init__` in every class, including `super().__init__()` calls |
| Exception Handling | `LibraryError` hierarchy (`ItemNotFoundError`, `ItemNotAvailableError`, `MemberNotFoundError`) |
| Clean code | Docstrings, properties, meaningful names, single-responsibility classes |
